In [10]:
import random
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [11]:
print("=" * 65)
print("STEP 1: COLLECTING MOVIE REVIEW DATA")
print("=" * 65)

positive_reviews = [
    "This movie was absolutely brilliant! The acting was superb and the storyline kept me hooked from start to finish.",
    "An outstanding film with breathtaking visuals and an emotional storyline. The director did an incredible job.",
    "Loved every moment of this movie. The characters were well-developed and the music was phenomenal.",
    "A masterpiece of cinema. The performances were exceptional and the screenplay was flawlessly executed.",
    "One of the best films I have seen this year. Gripping plot, amazing acting, and a stunning climax.",
    "The web series was binge-worthy! Every episode left me wanting more. Brilliant storytelling.",
    "Fantastic direction and superb cinematography. A must-watch for all movie lovers.",
    "The storyline was refreshingly original. The lead actor delivered a career-defining performance.",
    "Absolutely loved this film. It was funny, emotional, and visually stunning all at once.",
    "A delightful movie experience. Perfect blend of humor, drama, and action. Highly recommended!",
    "The music score elevated every scene. Combined with stellar performances, this film is unforgettable.",
    "Exceptional writing and top-notch acting. This movie will stay with me for a long time.",
    "A thrilling ride from start to finish. The twist at the end was mind-blowing. Best movie of the year.",
    "The ensemble cast was incredible. Every actor brought their A-game to this masterpiece.",
    "Visually stunning and emotionally powerful. The director's vision was realized perfectly on screen.",
    "I was completely engrossed from the opening scene. A truly brilliant piece of filmmaking.",
    "The chemistry between the lead actors was electric. The romantic storyline was beautifully portrayed.",
    "An inspiring and uplifting film that touched my heart. Wonderful performances by the entire cast.",
    "The action sequences were jaw-dropping. Combined with an intelligent script, this film delivers on all fronts.",
    "A cinematic gem. The pacing was perfect, the dialogues sharp, and the direction impeccable.",
    "This series had me on the edge of my seat. Superb production quality and a gripping narrative.",
    "A heartwarming story with brilliant acting. This movie deserves every award it can get.",
    "The special effects were mind-blowing and the story was deeply touching. A perfect movie.",
    "Outstanding performances and a captivating plot. This is what cinema is all about.",
    "Simply magical. The storytelling was poetic and the performances were breathtaking.",
]

negative_reviews = [
    "Terrible movie. The plot made no sense and the acting was wooden and unconvincing throughout.",
    "A complete waste of time. Poor screenplay, weak direction, and boring performances across the board.",
    "The worst movie I have seen in years. Predictable story, terrible acting, and awful music.",
    "Extremely disappointing. The trailer was misleading and the actual film was a total letdown.",
    "The storyline was confusing and the characters were completely underdeveloped. Very poor effort.",
    "Horrible direction and sloppy editing made this movie nearly unwatchable. Avoid at all costs.",
    "The web series dragged on endlessly with no plot progression. A complete bore from start to finish.",
    "Pathetic acting and a clichéd story. This film had nothing new to offer and was a total disaster.",
    "The special effects were laughably bad and the script was full of logical inconsistencies.",
    "One of the most boring films ever made. Nothing interesting happens for two and a half hours.",
    "The lead actor was stiff and unconvincing. The supporting cast fared no better in this disaster.",
    "Terrible screenplay and worse execution. This movie was a complete embarrassment to the industry.",
    "Utterly disappointing after all the hype. The film failed to deliver on any of its promises.",
    "The music was annoying, the acting was bad, and the story was a mess. Complete waste of money.",
    "Poor character development and a predictable ending made this movie a chore to sit through.",
    "The direction was amateurish and the production quality was shockingly low for a big-budget film.",
    "A painful watch. The dialogue was cringe-worthy and the plot made absolutely no sense whatsoever.",
    "The movie tried too hard to be clever but ended up being incomprehensible and boring.",
    "Absolutely dreadful. The pacing was off, the story was weak, and the acting was painful to watch.",
    "A soulless cash grab with no originality. The studio clearly had no faith in this project.",
    "The worst web series I have ever watched. Boring storyline and terrible character development.",
    "Disappointing on every level. The promising premise was completely wasted by poor execution.",
    "Awful dialogues and a predictable plot ruined what could have been an interesting concept.",
    "The film was tedious and uninspired. Two hours of my life I will never get back.",
    "Dismal performances and a convoluted plot made this one of the worst movies in recent memory.",
]

neutral_reviews = [
    "The movie was okay. Some scenes were interesting but overall it felt average and forgettable.",
    "A decent watch but nothing extraordinary. The acting was fine and the story was somewhat engaging.",
    "The film had its moments but also some significant weaknesses. An average cinematic experience.",
    "Not bad but not great either. The storyline was predictable but the performances were acceptable.",
    "A mediocre movie that neither impressed nor disappointed. Just an average film that passed time.",
    "The web series started well but lost steam midway. An uneven but watchable experience overall.",
    "Some good scenes are let down by an overall average script. Decent but could have been much better.",
    "The film was neither particularly good nor bad. It was a standard production with average everything.",
    "Watchable but forgettable. The cast did their best with a mediocre script and average direction.",
    "An adequate film with a few standout moments but nothing that truly elevates it above ordinary.",
    "The movie has some interesting ideas but fails to fully develop them. A mixed bag overall.",
    "Decent performances in a film that is ultimately let down by an uninspired and clichéd storyline.",
    "It is a passable film for a lazy afternoon. Nothing special but not terrible either.",
    "Average in every sense of the word. The film ticks all the boxes but brings nothing new to the table.",
    "A safe and unremarkable film. The direction was competent but lacked any real creative vision.",
    "The acting was decent but the inconsistent pacing and average script keep this from being great.",
    "An okay series to watch if you have nothing better to do. Some episodes were good, others were weak.",
    "Neither impressed nor bored by this film. It was a standard production that met basic expectations.",
    "The movie had potential but settled for being ordinary. A mixed and ultimately unsatisfying experience.",
    "Competently made but completely unoriginal. The film brings nothing fresh to a familiar genre.",
    "An average effort with some good moments. Not enough to make it memorable but not bad either.",
    "The film was adequate but instantly forgettable. It serves its purpose as light entertainment.",
    "Middling performances and a so-so script. The film is watchable but offers no real surprises.",
    "A fairly standard movie experience. Some things worked, others did not. Overall rather mediocre.",
    "The concept was interesting but the execution was average. Could have been much better or much worse.",
]

# Create DataFrame
all_reviews  = positive_reviews + negative_reviews + neutral_reviews
all_labels   = ["Positive"] * len(positive_reviews) + \
               ["Negative"] * len(negative_reviews) + \
               ["Neutral"]  * len(neutral_reviews)
all_movies   = (
    ["Inception 2", "Galaxy Quest", "The Grand Illusion", "Cosmic Journey",
     "The Last Stand", "Neon Nights S1", "Blue Horizon", "Storm Chaser",
     "The Final Act", "Laughter & Tears", "Echo Chamber", "The Reckoning",
     "Twist of Fate", "Ensemble", "Visionary", "First Light", "Chemistry",
     "Rising Sun", "Adrenaline Rush", "Perfect Storm", "Edge of Tomorrow S2",
     "The Golden Hour", "Spectacle", "Pure Cinema", "The Magic Hour"] +
    ["The Disaster Flick", "Wasted Potential", "The Worst Year", "Misleading Trailer",
     "Empty Characters", "Endless Boredom S1", "Sloppy Editing", "Cliché Fest",
     "Bad Effects", "Two Hours Lost", "Stiff Performance", "Script Disaster",
     "Overhyped Letdown", "Noise and Fury", "Flat Arc", "Amateur Night",
     "Cringe Dialogue", "Too Clever", "Dreadful Pacing", "Cash Grab",
     "Boring Series S1", "Broken Promises", "Awful Dialogue", "Tedium",
     "Dismal Memory"] +
    ["The Average Joe", "Decent Enough", "Middle Ground", "Predictable Path",
     "Mediocre Max", "Uneven Series S1", "Average Attempt", "Standard Issue",
     "Passable Pic", "Adequate Adventure", "Half Baked", "Decent Try",
     "Lazy Afternoon", "Box Ticker", "Safe Bet", "Inconsistent",
     "Okay Series S1", "Standard Fare", "Missed Potential", "Unoriginal",
     "Mixed Bag", "Light Fare", "So-So", "Forgettable", "Average Concept"]
)

df = pd.DataFrame({"review": all_reviews, "sentiment": all_labels, "movie": all_movies})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"  Total reviews collected : {len(df)}")
print(f"  Sentiment distribution  :")
print(df["sentiment"].value_counts().to_string())

STEP 1: COLLECTING MOVIE REVIEW DATA
  Total reviews collected : 75
  Sentiment distribution  :
sentiment
Positive    25
Neutral     25
Negative    25


In [12]:
print("\n" + "=" * 65)
print("STEP 2: PREPROCESSING REVIEW TEXT")
print("=" * 65)

stop_words  = set(stopwords.words("english"))
lemmatizer  = WordNetLemmatizer()

def preprocess(text: str) -> str:
    text  = text.lower()
    text  = re.sub(r"[^a-z\s]", "", text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return " ".join(tokens)

df["clean_review"] = df["review"].apply(preprocess)
df["word_count"]   = df["clean_review"].apply(lambda x: len(x.split()))

print("  Sample preprocessed review:")
print(f"    ORIGINAL : {df['review'].iloc[0][:80]}...")
print(f"    CLEANED  : {df['clean_review'].iloc[0][:80]}...")
print(f"\n  Avg word count after preprocessing: {df['word_count'].mean():.1f}")


STEP 2: PREPROCESSING REVIEW TEXT
  Sample preprocessed review:
    ORIGINAL : One of the best films I have seen this year. Gripping plot, amazing acting, and ...
    CLEANED  : one best film seen year gripping plot amazing acting stunning climax...

  Avg word count after preprocessing: 8.7


In [13]:
print("\n" + "=" * 65)
print("STEP 3: BUILDING SENTIMENT CLASSIFICATION MODEL")
print("=" * 65)

label_map = {"Positive": 2, "Neutral": 1, "Negative": 0}
df["label"] = df["sentiment"].map(label_map)

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"], df["label"], test_size=0.25, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Naive Bayes":         MultinomialNB(alpha=0.5),
    "Linear SVM":          LinearSVC(C=1.0, max_iter=2000, random_state=42),
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=5000,
                                               sublinear_tf=True)), ("clf", clf)])
    cv_scores = cross_val_score(pipe, df["clean_review"], df["label"], cv=5, scoring="f1_macro")
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average="macro")
    results[name] = {"pipeline": pipe, "accuracy": acc, "f1": f1,
                     "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std(),
                     "y_pred": y_pred}
    print(f"  {name:<22} | Acc: {acc:.3f} | F1: {f1:.3f} | CV: {cv_scores.mean():.3f}±{cv_scores.std():.3f}")

best_name = max(results, key=lambda k: results[k]["f1"])
best      = results[best_name]
print(f"\n  ✓ Best model: {best_name} (F1 = {best['f1']:.3f})")

print("\n  Classification Report (Best Model):")
inv_map = {v: k for k, v in label_map.items()}
print(classification_report(y_test, best["y_pred"],
      target_names=["Negative", "Neutral", "Positive"]))

# Store predictions on full dataset
df["predicted_sentiment"] = best["pipeline"].predict(df["clean_review"])
df["predicted_label"]     = df["predicted_sentiment"].map(inv_map)



STEP 3: BUILDING SENTIMENT CLASSIFICATION MODEL
  Logistic Regression    | Acc: 0.789 | F1: 0.786 | CV: 0.778±0.129
  Naive Bayes            | Acc: 0.789 | F1: 0.786 | CV: 0.753±0.142
  Linear SVM             | Acc: 0.895 | F1: 0.897 | CV: 0.764±0.145

  ✓ Best model: Linear SVM (F1 = 0.897)

  Classification Report (Best Model):
              precision    recall  f1-score   support

    Negative       0.83      0.83      0.83         6
     Neutral       1.00      1.00      1.00         6
    Positive       0.86      0.86      0.86         7

    accuracy                           0.89        19
   macro avg       0.90      0.90      0.90        19
weighted avg       0.89      0.89      0.89        19



In [14]:
print("\n" + "=" * 65)
print("STEP 4: ANALYZING AUDIENCE OPINIONS")
print("=" * 65)

sent_counts = df["predicted_label"].value_counts()
print("\n  Predicted Sentiment Distribution:")
for sent, cnt in sent_counts.items():
    pct = cnt / len(df) * 100
    bar = "█" * int(pct / 3)
    print(f"    {sent:<10}: {cnt:>3} ({pct:5.1f}%)  {bar}")

pos_words = " ".join(df[df["predicted_label"] == "Positive"]["clean_review"])
neg_words = " ".join(df[df["predicted_label"] == "Negative"]["clean_review"])

top_pos = Counter(pos_words.split()).most_common(10)
top_neg = Counter(neg_words.split()).most_common(10)

print("\n  Top Positive Keywords:", [w for w, _ in top_pos])
print("  Top Negative Keywords:", [w for w, _ in top_neg])


STEP 4: ANALYZING AUDIENCE OPINIONS

  Predicted Sentiment Distribution:
    Positive  :  25 ( 33.3%)  ███████████
    Neutral   :  25 ( 33.3%)  ███████████
    Negative  :  25 ( 33.3%)  ███████████

  Top Positive Keywords: ['film', 'movie', 'performance', 'every', 'acting', 'brilliant', 'storyline', 'stunning', 'superb', 'direction']
  Top Negative Keywords: ['movie', 'made', 'film', 'plot', 'story', 'acting', 'boring', 'poor', 'terrible', 'complete']


In [15]:
print("\n" + "=" * 65)
print("STEP 5: VISUALIZING SENTIMENT RESULTS")
print("=" * 65)

COLORS = {"Positive": "#2ecc71", "Negative": "#e74c3c", "Neutral": "#f39c12"}
PALETTE = [COLORS["Negative"], COLORS["Neutral"], COLORS["Positive"]]
sns.set_style("whitegrid")

fig = plt.figure(figsize=(20, 24))
fig.patch.set_facecolor("#0d1117")
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

title_style = dict(fontsize=11, fontweight="bold", color="white", pad=10)
axis_style  = dict(color="#aaaaaa", fontsize=9)


STEP 5: VISUALIZING SENTIMENT RESULTS


In [17]:
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor("#161b22")
sizes  = [sent_counts.get(s, 0) for s in ["Positive", "Negative", "Neutral"]]
colors = [COLORS[s] for s in ["Positive", "Negative", "Neutral"]]
wedges, texts, autotexts = ax1.pie(
    sizes, labels=["Positive", "Negative", "Neutral"],
    colors=colors, autopct="%1.1f%%", startangle=140,
    textprops={"color": "white", "fontsize": 9},
    wedgeprops={"edgecolor": "#0d1117", "linewidth": 2})
for at in autotexts:
    at.set_fontweight("bold")
ax1.set_title("Sentiment Distribution", **title_style)

Text(0.5, 1.0, 'Sentiment Distribution')

In [18]:
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor("#161b22")
sentiments = ["Positive", "Neutral", "Negative"]
counts     = [sent_counts.get(s, 0) for s in sentiments]
bars = ax2.bar(sentiments, counts,
               color=[COLORS[s] for s in sentiments],
               edgecolor="#0d1117", linewidth=1.5, width=0.5)
for bar, cnt in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(cnt), ha="center", color="white", fontweight="bold", fontsize=10)
ax2.set_facecolor("#161b22")
ax2.tick_params(colors="#aaaaaa")
ax2.spines[:].set_color("#333333")
ax2.set_title("Review Count by Sentiment", **title_style)
ax2.set_ylabel("Count", **axis_style)


Text(0, 0.5, 'Count')

In [19]:
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor("#161b22")
model_names = list(results.keys())
accs = [results[m]["accuracy"] for m in model_names]
f1s  = [results[m]["f1"]       for m in model_names]
x    = np.arange(len(model_names))
w    = 0.35
ax3.bar(x - w/2, accs, w, label="Accuracy", color="#3498db", edgecolor="#0d1117")
ax3.bar(x + w/2, f1s,  w, label="F1-Score", color="#9b59b6", edgecolor="#0d1117")
ax3.set_xticks(x)
ax3.set_xticklabels(["LR", "NB", "SVM"], color="#aaaaaa", fontsize=9)
ax3.set_ylim(0, 1.15)
ax3.tick_params(colors="#aaaaaa")
ax3.spines[:].set_color("#333333")
ax3.legend(facecolor="#161b22", labelcolor="white", fontsize=8)
ax3.set_title("Model Performance Comparison", **title_style)
ax3.set_ylabel("Score", **axis_style)
for bar in ax3.patches:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{bar.get_height():.2f}", ha="center", color="white", fontsize=7)


In [20]:
ax4 = fig.add_subplot(gs[1, 0])
ax4.set_facecolor("#161b22")
cm = confusion_matrix(y_test, best["y_pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Neg", "Neu", "Pos"],
            yticklabels=["Neg", "Neu", "Pos"],
            ax=ax4, cbar_kws={"shrink": 0.8},
            linewidths=0.5, linecolor="#0d1117")
ax4.set_title(f"Confusion Matrix\n({best_name})", **title_style)
ax4.set_xlabel("Predicted", **axis_style)
ax4.set_ylabel("Actual",    **axis_style)
ax4.tick_params(colors="#aaaaaa")


In [21]:
ax5 = fig.add_subplot(gs[1, 1])
ax5.set_facecolor("#161b22")
for sent in ["Positive", "Negative", "Neutral"]:
    data = df[df["predicted_label"] == sent]["word_count"]
    ax5.hist(data, bins=10, alpha=0.7, color=COLORS[sent],
             edgecolor="#0d1117", label=sent)
ax5.set_title("Word Count Distribution", **title_style)
ax5.set_xlabel("Words per Review", **axis_style)
ax5.set_ylabel("Frequency", **axis_style)
ax5.legend(facecolor="#161b22", labelcolor="white", fontsize=8)
ax5.tick_params(colors="#aaaaaa")
ax5.spines[:].set_color("#333333")


In [22]:
ax6 = fig.add_subplot(gs[1, 2])
ax6.set_facecolor("#161b22")
cv_means = [results[m]["cv_mean"] for m in model_names]
cv_stds  = [results[m]["cv_std"]  for m in model_names]
bars6 = ax6.barh(["LR", "NB", "SVM"], cv_means,
                 xerr=cv_stds, color=["#3498db","#2ecc71","#e67e22"],
                 edgecolor="#0d1117", height=0.4,
                 error_kw={"ecolor": "white", "capsize": 4})
ax6.set_xlim(0, 1.1)
ax6.tick_params(colors="#aaaaaa")
ax6.spines[:].set_color("#333333")
ax6.set_title("5-Fold CV F1 Scores", **title_style)
ax6.set_xlabel("F1 Score", **axis_style)
for bar, mean in zip(bars6, cv_means):
    ax6.text(mean + 0.02, bar.get_y() + bar.get_height()/2,
             f"{mean:.3f}", va="center", color="white", fontsize=8)



In [23]:
ax7 = fig.add_subplot(gs[2, 0:2])
ax7.set_facecolor("#161b22")
if pos_words.strip():
    wc = WordCloud(width=800, height=300, background_color="#161b22",
                   colormap="Greens", max_words=60,
                   collocations=False).generate(pos_words)
    ax7.imshow(wc, interpolation="bilinear")
ax7.axis("off")
ax7.set_title("☀  Positive Review Word Cloud", **title_style)


Text(0.5, 1.0, '☀  Positive Review Word Cloud')

In [24]:
ax8 = fig.add_subplot(gs[2, 2])
ax8.set_facecolor("#161b22")
if neg_words.strip():
    wc_neg = WordCloud(width=400, height=300, background_color="#161b22",
                       colormap="Reds", max_words=40,
                       collocations=False).generate(neg_words)
    ax8.imshow(wc_neg, interpolation="bilinear")
ax8.axis("off")
ax8.set_title("☁  Negative Word Cloud", **title_style)


Text(0.5, 1.0, '☁  Negative Word Cloud')

In [25]:
ax9 = fig.add_subplot(gs[3, 0])
ax9.set_facecolor("#161b22")
pos_df = pd.DataFrame(top_pos, columns=["word", "freq"]).sort_values("freq")
ax9.barh(pos_df["word"], pos_df["freq"], color="#2ecc71", edgecolor="#0d1117")
ax9.set_title("Top Positive Keywords", **title_style)
ax9.set_xlabel("Frequency", **axis_style)
ax9.tick_params(colors="#aaaaaa")
ax9.spines[:].set_color("#333333")

ax10 = fig.add_subplot(gs[3, 1])
ax10.set_facecolor("#161b22")
neg_df = pd.DataFrame(top_neg, columns=["word", "freq"]).sort_values("freq")
ax10.barh(neg_df["word"], neg_df["freq"], color="#e74c3c", edgecolor="#0d1117")
ax10.set_title("Top Negative Keywords", **title_style)
ax10.set_xlabel("Frequency", **axis_style)
ax10.tick_params(colors="#aaaaaa")
ax10.spines[:].set_color("#333333")



In [26]:
ax11 = fig.add_subplot(gs[3, 2])
ax11.set_facecolor("#161b22")
ring_data   = [sent_counts.get(s, 0) for s in ["Positive", "Neutral", "Negative"]]
ring_colors = [COLORS[s] for s in ["Positive", "Neutral", "Negative"]]
wedges2, _, autotexts2 = ax11.pie(
    ring_data, colors=ring_colors, autopct="%1.0f%%", startangle=90,
    pctdistance=0.75,
    textprops={"color": "white", "fontsize": 8},
    wedgeprops={"edgecolor": "#0d1117", "linewidth": 2, "width": 0.5})
for at in autotexts2:
    at.set_fontweight("bold")
centre = plt.Circle((0, 0), 0.45, fc="#161b22")
ax11.add_patch(centre)
ax11.text(0, 0, f"{round(sent_counts.get('Positive',0)/len(df)*100)}%\nPos",
          ha="center", va="center", color="#2ecc71", fontsize=10, fontweight="bold")
ax11.set_title("Audience Sentiment Ring", **title_style)
ax11.legend(["Positive","Neutral","Negative"],
            facecolor="#161b22", labelcolor="white",
            loc="lower center", fontsize=7, ncol=3)


In [28]:
import os

fig.suptitle("🎬  NLP Movie Review Sentiment Classification Dashboard",
             fontsize=16, fontweight="bold", color="white", y=0.995)

out_path = "/mnt/user-data/outputs/movie_sentiment_dashboard.png"

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(out_path), exist_ok=True)

plt.savefig(out_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  ✓ Dashboard saved → {out_path}")

  ✓ Dashboard saved → /mnt/user-data/outputs/movie_sentiment_dashboard.png


In [29]:
print("\n" + "=" * 65)
print("PROJECT SUMMARY")
print("=" * 65)
print(f"  Dataset size           : {len(df)} movie reviews")
print(f"  Train / Test split     : {len(X_train)} / {len(X_test)} reviews")
print(f"  Best model             : {best_name}")
print(f"  Accuracy               : {best['accuracy']:.3f}  ({best['accuracy']*100:.1f}%)")
print(f"  Macro F1-Score         : {best['f1']:.3f}")
print(f"  Positive reviews       : {sent_counts.get('Positive', 0)}")
print(f"  Negative reviews       : {sent_counts.get('Negative', 0)}")
print(f"  Neutral  reviews       : {sent_counts.get('Neutral',  0)}")
print("=" * 65)
print("ALL TASKS COMPLETED SUCCESSFULLY ✓")




PROJECT SUMMARY
  Dataset size           : 75 movie reviews
  Train / Test split     : 56 / 19 reviews
  Best model             : Linear SVM
  Accuracy               : 0.895  (89.5%)
  Macro F1-Score         : 0.897
  Positive reviews       : 25
  Negative reviews       : 25
  Neutral  reviews       : 25
ALL TASKS COMPLETED SUCCESSFULLY ✓
